In [ ]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import os, subprocess
import gc
bucket = os.getenv('WORKSPACE_BUCKET')
amia_path = f"{bucket}/amia"

print(f"data downloaded to: {amia_path}")

In [ ]:
subprocess.run(["gsutil", "ls", f"{bucket}/amia/"])

In [ ]:
!df -h

In [ ]:
def save_large_df_to_bucket(df, bucket_path, local_file="temp.csv", chunksize=500000):
    bucket = os.getenv("WORKSPACE_BUCKET")
    n = len(df)
    num_chunks = max(1, n // chunksize + (1 if n % chunksize else 0))

    print(f"downloading: {local_file}")
    with open(local_file, "w") as f:
        for i, chunk in enumerate(
            tqdm(np.array_split(df, num_chunks), total=num_chunks, desc="Writing CSV")
        ):
            chunk.to_csv(f, index=False, header=(i == 0), mode="a")
    subprocess.run(["gsutil", "cp", local_file, f"{bucket}/{bucket_path}"], check=True)
    
    os.remove(local_file)
    print(f"uploaded successfully: {bucket}/{bucket_path}")

In [ ]:
import pandas
import os

# This query represents dataset "amia_wOPRx_dx_Positive" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_07210793_condition_sql = """
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_source_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (1326498, 1568106, 1568107, 1568111, 1568113, 1568115, 37402463, 44824126, 44829933, 44835800, 44836978, 44836979, 45533067, 45533068, 45542792, 45542793, 45542794, 45542795, 45542796, 45542797, 45542798, 45557159, 45557160, 45557162, 45562007, 45566782, 45566784, 45566785, 45566786, 45566787, 45571712, 45571713, 45571714, 45576499, 45576500, 45576501, 45581409, 45581411, 45586193, 45586196, 45586197, 45586198, 45591090, 45591091, 45591095, 45595854, 45595856, 45595857, 45600697, 45605458, 45605459, 45605460, 45605461, 45605464, 725281)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 0 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (21604254)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria ) ))) c_occurrence 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id"""

dataset_07210793_condition_df = pd.read_gbq(
    dataset_07210793_condition_sql, 
    dialect="standard"
)
save_large_df_to_bucket(dataset_07210793_condition_df, "amia/positive.csv")
del dataset_07210793_condition_df
gc.collect()
display(pd.read_csv(f"{amia_path}/postive.csv", nrows=5))

In [ ]:
import pandas
import os

# This query represents dataset "amia_wOPRx_dx_Positive" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_43847808_condition_sql = """
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_source_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (1326498, 1568106, 1568107, 1568111, 1568113, 1568115, 37402463, 44824126, 44829933, 44835800, 44836978, 44836979, 45533067, 45533068, 45542792, 45542793, 45542794, 45542795, 45542796, 45542797, 45542798, 45557159, 45557160, 45557162, 45562007, 45566782, 45566784, 45566785, 45566786, 45566787, 45571712, 45571713, 45571714, 45576499, 45576500, 45576501, 45581409, 45581411, 45586193, 45586196, 45586197, 45586198, 45591090, 45591091, 45591095, 45595854, 45595856, 45595857, 45600697, 45605458, 45605459, 45605460, 45605461, 45605464, 725281)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 0 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (21604254)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria ) ))) c_occurrence 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id"""

dataset_43847808_condition_df = pandas.read_gbq(
    dataset_43847808_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_43847808_condition_df.head(5)

In [ ]:
dataset_43847808_condition_df.to_csv(f"{bucket}/amia/positive.csv", index=False)

In [ ]:
del dataset_43847808_condition_df
gc.collect()

In [ ]:
import pandas
import os

# This query represents dataset "observation_amia" for domain "observation" and was generated for All of Us Controlled Tier Dataset v8
dataset_80365746_observation_sql = """
    SELECT
        observation.person_id,
        o_standard_concept.concept_name as standard_concept_name,
        o_standard_concept.concept_code as standard_concept_code,
        o_standard_concept.vocabulary_id as standard_vocabulary,
        observation.observation_datetime,
        observation.value_as_number,
        observation.value_as_string,
        observation.value_as_concept_id,
        o_value.concept_name as value_as_concept_name,
        observation.qualifier_concept_id,
        o_qualifier.concept_name as qualifier_concept_name,
        observation.unit_concept_id,
        o_unit.concept_name as unit_concept_name,
        o_visit.concept_name as visit_occurrence_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.observation` observation 
        WHERE
            (
                observation_concept_id IN (2617831, 2617910, 3043579, 4022240, 4027634, 4052625, 4073163, 4139934, 4154624, 4235714, 4251171, 4305297, 43054909, 4322482, 43533855, 45766930)
            )  
            AND (
                observation.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (21604254)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria ) ))) observation 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` o_standard_concept 
                ON observation.observation_concept_id = o_standard_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` o_value 
                ON observation.value_as_concept_id = o_value.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` o_qualifier 
                ON observation.qualifier_concept_id = o_qualifier.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` o_unit 
                ON observation.unit_concept_id = o_unit.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                ON observation.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` o_visit 
                ON v.visit_concept_id = o_visit.concept_id"""

dataset_80365746_observation_df = pd.read_gbq(
    dataset_80365746_observation_sql, 
    dialect="standard"
)
save_large_df_to_bucket(dataset_80365746_observation_df, "amia/observations.csv")
del dataset_80365746_observation_df
gc.collect()
display(pd.read_csv(f"{amia_path}/observations.csv", nrows=5))

In [ ]:
dataset_80365746_observation_df.to_csv(f"{bucket}/amia/observations.csv", index=False)

In [ ]:
del dataset_80365746_observation_df
gc.collect()

In [ ]:
import os
import gc
import subprocess
import pandas as pd
from google.cloud import bigquery


def export_bigquery_result_to_bucket(
    sql_query,
    file_name="labs",
    bucket_subdir="amia",
    local_dir=".",
    preview_nrows=5,
    dry_run_only=False,
):
    """
    Stream BigQuery query results chunk by chunk to a local CSV,
    then upload to GCS bucket, with row-count validation.

    Parameters
    ----------
    sql_query : str
        The SELECT query to export.
    file_name : str
        Output file stem, e.g. "labs" -> labs.csv
    bucket_subdir : str
        Folder inside WORKSPACE_BUCKET, e.g. "amia"
    local_dir : str
        Local temp directory for intermediate csv
    preview_nrows : int
        Number of rows to preview locally after writing
    dry_run_only : bool
        If True, do everything except gsutil upload
    """

    client = bigquery.Client()
    bucket = os.getenv("WORKSPACE_BUCKET")
    if not bucket:
        raise EnvironmentError("WORKSPACE_BUCKET is not set.")

    os.makedirs(local_dir, exist_ok=True)

    local_file = os.path.join(local_dir, f"{file_name}_temp.csv")
    gcs_path = f"{bucket.rstrip('/')}/{bucket_subdir.strip('/')}/{file_name}.csv"

    print("=" * 80)
    print(f"[INFO] file_name      : {file_name}")
    print(f"[INFO] local_file     : {local_file}")
    print(f"[INFO] bucket target  : {gcs_path}")
    print("=" * 80)
    
    count_sql = f"""
    SELECT COUNT(*) AS n
    FROM (
        {sql_query}
    )
    """

    print("[INFO] Counting rows from query...")
    count_job = client.query(count_sql)
    count_df = count_job.result().to_dataframe()
    expected_rows = int(count_df.iloc[0]["n"])
    print(f"[INFO] Expected rows = {expected_rows:,}")

    if expected_rows == 0:
        print("[WARN] Query returned 0 rows. No file will be written.")
        return {
            "expected_rows": 0,
            "written_rows": 0,
            "gcs_path": gcs_path,
            "local_file": None,
        }

    print("[INFO] Starting streaming export...")
    query_job = client.query(sql_query)

    total_written = 0
    chunk_count = 0
    header_written = False

    try:
        row_iter = query_job.result().to_dataframe_iterable()

        for chunk_idx, chunk in enumerate(row_iter, start=1):
            n_rows = len(chunk)

            if n_rows == 0:
                print(f"[WARN] chunk {chunk_idx} is empty, skipped.")
                continue

            mode = "w" if not header_written else "a"
            header = not header_written

            chunk.to_csv(
                local_file,
                index=False,
                mode=mode,
                header=header,
            )

            header_written = True
            chunk_count += 1
            total_written += n_rows

            print(
                f"[INFO] chunk={chunk_idx:,} | "
                f"rows={n_rows:,} | "
                f"total_written={total_written:,}"
            )

            del chunk
            gc.collect()

    except Exception as e:
        print("[ERROR] Export failed during chunk writing.")
        print(f"[ERROR] Last completed chunk_count = {chunk_count:,}")
        print(f"[ERROR] Rows written so far      = {total_written:,}")
        raise RuntimeError(f"Streaming export failed: {e}") from e

    print("=" * 80)
    print(f"[INFO] Finished local writing.")
    print(f"[INFO] total chunks written = {chunk_count:,}")
    print(f"[INFO] total rows written   = {total_written:,}")
    print("=" * 80)

    print("[INFO] Validating local CSV row count...")
    with open(local_file, "r", encoding="utf-8", errors="ignore") as f:
        line_count = sum(1 for _ in f)

    actual_csv_rows = max(0, line_count - 1)

    print(f"[INFO] Local CSV data rows = {actual_csv_rows:,}")

    if total_written != expected_rows:
        raise ValueError(
            f"Mismatch between expected_rows ({expected_rows:,}) "
            f"and total_written ({total_written:,})."
        )

    if actual_csv_rows != expected_rows:
        raise ValueError(
            f"Mismatch between expected_rows ({expected_rows:,}) "
            f"and local CSV rows ({actual_csv_rows:,})."
        )

    print("[INFO] Row-count validation passed.")

    print(f"[INFO] Preview first {preview_nrows} rows from local CSV:")
    preview_df = pd.read_csv(local_file, nrows=preview_nrows)
    display(preview_df)

    if dry_run_only:
        print("[INFO] dry_run_only=True, skipping gsutil upload.")
    else:
        print("[INFO] Uploading to bucket...")
        subprocess.run(["gsutil", "cp", local_file, gcs_path], check=True)
        print(f"[INFO] Uploaded to {gcs_path}")

        subprocess.run(["gsutil", "ls", "-lh", gcs_path], check=False)

    if os.path.exists(local_file):
        os.remove(local_file)
        print(f"[INFO] Removed local temp file: {local_file}")

    return {
        "expected_rows": expected_rows,
        "written_rows": total_written,
        "gcs_path": gcs_path,
        "local_file": local_file,
    }

In [ ]:
export_bigquery_result_to_bucket(
    dataset_95277459_measurement_sql,
    file_name="labs",
    bucket_subdir="amia",
)

In [ ]:
import pandas
import os

# This query represents dataset "labs_amia" for domain "measurement" and was generated for All of Us Controlled Tier Dataset v8
dataset_95277459_measurement_sql = """
    SELECT
        measurement.person_id,
        m_standard_concept.concept_name as standard_concept_name,
        m_standard_concept.concept_code as standard_concept_code,
        m_standard_concept.vocabulary_id as standard_vocabulary,
        measurement.measurement_datetime,
        measurement.value_as_number,
        measurement.value_as_concept_id,
        m_value.concept_name as value_as_concept_name,
        measurement.unit_concept_id,
        m_unit.concept_name as unit_concept_name,
        measurement.range_low,
        measurement.range_high,
        m_visit.concept_name as visit_occurrence_concept_name,
        measurement.measurement_source_concept_id,
        m_source_concept.concept_name as source_concept_name,
        m_source_concept.concept_code as source_concept_code,
        m_source_concept.vocabulary_id as source_vocabulary,
        measurement.unit_source_value,
        measurement.value_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.measurement` measurement 
        WHERE
            (
                measurement_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (3000330, 3000571, 3000593, 3002148, 3002364, 3002730, 3003985, 3004037, 3004410, 3004574, 3005167, 3005479, 3006322, 3006923, 3006932, 3007070, 3007220, 3007463, 3007682, 3007785, 3008152, 3008727, 3008776, 3009337, 3009596, 3009956, 3009984, 3010822, 3011302, 3011402, 3012131, 3012659, 3012920, 3013184, 3013682, 3013721, 3013814, 3014032, 3014111, 3014798, 3015208, 3016723, 3018060, 3018335, 3018965, 3019535, 3019789, 3019800, 3019911, 3019977, 3020216, 3020231, 3020293, 3020460, 3020630, 3020891, 3021016, 3021043, 3021332, 3021337, 3021461, 3021601, 3021999, 3022192, 3022217, 3022551, 3022755, 3023083, 3023368, 3023917, 3024128, 3024171, 3024561, 3024741, 3024789, 3024929, 3025644, 3025770, 3026363, 3026408, 3026625, 3027114, 3027281, 3027597, 3027801, 3027944, 3027946, 3028140, 3028288, 3028741, 3031029, 3031090, 3032008, 3032515, 3033543, 3033667, 3034204, 3034426, 3034962, 3035722, 3035995, 3036180, 3036541, 3036987, 3037242, 3038160, 3038276,
 3043411, 3044491, 3045425, 3047181, 3052141, 36303370, 36662059, 37020246, 37022664, 37049000, 37051076, 37061053, 37062219, 40761463, 40762499, 40762735, 40764183, 40764186, 40768496, 4197602, 43055653, 43055655, 4319466, 46234777, 46237030)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                measurement.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (21604254)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria ) ))) measurement 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_standard_concept 
                ON measurement.measurement_concept_id = m_standard_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_value 
                ON measurement.value_as_concept_id = m_value.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_unit 
                ON measurement.unit_concept_id = m_unit.concept_id 
        LEFT JOIn
            `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                ON measurement.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_visit 
                ON v.visit_concept_id = m_visit.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_source_concept 
                ON measurement.measurement_source_concept_id = m_source_concept.concept_id"""

import os, gc, subprocess
from google.cloud import bigquery

def rescue_measurement_export(sql_query, file_name="labs"):
    client = bigquery.Client()
    bucket = os.getenv("WORKSPACE_BUCKET")
    local_file = f"./{file_name}_temp.csv"
    cloud_path = f"amia/{file_name}.csv"
    query_job = client.query(sql_query)
    chunks = query_job.result().to_dataframe_iterable()

    print(f"downloading {file_name}")
    
    for i, chunk in enumerate(chunks):
        mode = 'w' if i == 0 else 'a'
        header = True if i == 0 else False
        chunk.to_csv(local_file, index=False, mode=mode, header=header)
        del chunk
        gc.collect()
        
        if (i + 1) % 5 == 0:
            print(f"finished {i + 1} packages...")

    print("uploading to Bucket...")
    subprocess.run(["gsutil", "cp", local_file, f"{bucket}/{cloud_path}"], check=True)
    if os.path.exists(local_file):
        os.remove(local_file)
    print(f"saved to {bucket}/{cloud_path}")

In [ ]:
import pandas
import os

# This query represents dataset "Physical Measurements_amia" for domain "measurement" and was generated for All of Us Controlled Tier Dataset v8
dataset_51898356_measurement_sql = """
    SELECT
        measurement.person_id,
        m_standard_concept.concept_name as standard_concept_name,
        m_standard_concept.concept_code as standard_concept_code,
        m_standard_concept.vocabulary_id as standard_vocabulary,
        measurement.measurement_datetime,
        m_type.concept_name as measurement_type_concept_name,
        measurement.value_as_number,
        measurement.value_as_concept_id,
        m_value.concept_name as value_as_concept_name,
        measurement.unit_concept_id,
        m_unit.concept_name as unit_concept_name,
        measurement.range_low,
        measurement.range_high,
        m_visit.concept_name as visit_occurrence_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.measurement` measurement 
        WHERE
            (
                measurement_source_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (1586218, 903115, 903118, 903121, 903124, 903126, 903133, 903135, 903136)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 0 
                    AND is_selectable = 1)
            )  
            AND (
                measurement.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (21604254)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria ) ))) measurement 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_standard_concept 
                ON measurement.measurement_concept_id = m_standard_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_type 
                ON measurement.measurement_type_concept_id = m_type.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_value 
                ON measurement.value_as_concept_id = m_value.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_unit 
                ON measurement.unit_concept_id = m_unit.concept_id 
        LEFT JOIn
            `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                ON measurement.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_visit 
                ON v.visit_concept_id = m_visit.concept_id"""

dataset_51898356_measurement_df = pandas.read_gbq(
    dataset_51898356_measurement_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")
save_large_df_to_bucket(dataset_51898356_measurement_df, "amia/measurements.csv")
del dataset_51898356_measurement_df
gc.collect()
display(pd.read_csv(f"{amia_path}/measurements.csv", nrows=5))

In [ ]:
import pandas
import os

# This query represents dataset "related_procedure_amia" for domain "procedure" and was generated for All of Us Controlled Tier Dataset v8
dataset_23700655_procedure_sql = """
    SELECT
        procedure.person_id,
        p_standard_concept.concept_name as standard_concept_name,
        p_standard_concept.concept_code as standard_concept_code,
        p_standard_concept.vocabulary_id as standard_vocabulary,
        procedure.procedure_datetime,
        p_visit.concept_name as visit_occurrence_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.procedure_occurrence` procedure 
        WHERE
            (
                procedure_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (4108316, 4210480, 4230911, 4261832, 4327941)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                procedure.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (21604254)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria ) ))) procedure 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_standard_concept 
                ON procedure.procedure_concept_id = p_standard_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                ON procedure.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_visit 
                ON v.visit_concept_id = p_visit.concept_id"""

export_bigquery_result_to_bucket(
    sql_query=dataset_23700655_procedure_sql, 
    file_name="procedures"
)



In [ ]:
import pandas
import os

# This query represents dataset "survey_amia" for domain "survey" and was generated for All of Us Controlled Tier Dataset v8
dataset_36000796_survey_sql = """
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.survey,
        answer.question,
        answer.answer  
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.ds_survey` answer   
    WHERE
        (
            question_concept_id IN (1384490, 1384518, 1384625, 1585370, 1585375, 1585386, 1585389, 1585402, 1585711, 1585717, 1585723, 1585729, 1585735, 1585741, 1585747, 1585754, 1585760, 1585838, 1585845, 1585852, 1585879, 1585886, 1585889, 1585892, 1585899, 1585940, 1585952, 1586150, 1586151, 1586152, 1586153, 1586154, 1586155, 1586156, 43528663, 43528665, 43528841, 43528842, 43528843, 43528847, 43529899, 43529903, 43529904, 43529905, 43529906, 43529976, 43529977, 43530268, 43530356, 43530357, 43530358, 43530362, 43530409, 43530410, 43530411, 43530412, 43530413, 43530415, 43530416, 43530417, 43530418, 43530437, 43530438, 43530439, 43530514, 43530515, 43530516, 43530520, 43530557, 43530559, 43530562, 43530583, 43530584, 43530585, 43530588, 43530593, 43530594, 43530595, 836803, 836804, 836805, 836827, 836850, 903575, 903578) 
            OR question_concept_id IN (SELECT
                DISTINCT concept_id                         
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c                         
            JOIN
                (SELECT
                    CAST(cr.id as string) AS id                               
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr                               
                WHERE
                    concept_id IN (1585855, 40192389)                               
                    AND domain_id = 'SURVEY') a 
                    ON (c.path like CONCAT('%', a.id, '.%'))                         
            WHERE
                domain_id = 'SURVEY'                         
                AND type = 'PPI'                         
                AND subtype = 'QUESTION')
        )  
        AND (
            answer.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    criteria.person_id 
                FROM
                    (SELECT
                        DISTINCT person_id, entry_date, concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                    WHERE
                        (concept_id IN(SELECT
                            DISTINCT ca.descendant_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                        JOIN
                            (SELECT
                                DISTINCT c.concept_id       
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id             
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                WHERE
                                    concept_id IN (21604254)             
                                    AND full_text LIKE '%_rank1]%'       ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) b 
                                ON (ca.ancestor_id = b.concept_id)) 
                            AND is_standard = 1)) criteria ) ))"""

export_bigquery_result_to_bucket(
    sql_query=dataset_36000796_survey_sql, 
    file_name="surveys"
)

In [ ]:
import pandas
import os

# This query represents dataset "drug_amia" for domain "drug" and was generated for All of Us Controlled Tier Dataset v8
dataset_37661123_drug_sql = """
    SELECT
        d_exposure.person_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_standard_concept.concept_code as standard_concept_code,
        d_exposure.drug_exposure_start_datetime 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.drug_exposure` d_exposure 
        WHERE
            (
                drug_concept_id IN (SELECT
                    DISTINCT ca.descendant_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                JOIN
                    (SELECT
                        DISTINCT c.concept_id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id             
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                        WHERE
                            concept_id IN (21603248, 21603444, 21603932, 21604069, 21604253, 21604489, 21604685, 21604816)             
                            AND full_text LIKE '%_rank1]%'       ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) b 
                        ON (ca.ancestor_id = b.concept_id)))  
                    AND (d_exposure.PERSON_ID IN (SELECT
                        distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (21604254)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria ) ))) d_exposure 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_standard_concept 
                ON d_exposure.drug_concept_id = d_standard_concept.concept_id"""

export_bigquery_result_to_bucket(
    sql_query=dataset_37661123_drug_sql, 
    file_name="drugs"
)

In [ ]:
!gsutil ls -lh {os.getenv("WORKSPACE_BUCKET")}/amia/*.csv

In [ ]:
import pandas
import os

# This query represents dataset "amia condition" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_18235021_condition_sql = """
    SELECT
        c_occurrence.person_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_type.concept_name as condition_type_concept_name,
        visit.concept_name as visit_occurrence_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_source_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (1326497, 1326500, 1326501, 1326502, 1326503, 1326505, 1326896, 1326897, 1326898, 1326899, 1567377, 1567379, 1567844, 1567849, 1567850, 1568096, 1568097, 1568100, 1568102, 1568129, 1568142, 1568146, 1568149, 1568153, 1568157, 1568165, 1568177, 1568178, 1568179, 1568180, 1568206, 1568207, 1568208, 1568209, 1568211, 1568213, 1568214, 1568217, 1568218, 1568219, 1568232, 1568233, 1568420, 1568421, 1568422, 1569155, 1569487, 1569719, 1569722, 1569739, 1569740, 1570701, 1570703, 1570704, 1570705, 1571057, 1571059, 1571060, 1571061, 1571062, 1571063, 1571064, 1574685, 1576268, 1595617, 1595618, 1595619, 1595620, 17836, 17839, 17841, 17858, 35205769, 35205770, 35205771, 35205772, 35205773, 35205774, 35205775, 35205776, 35206724, 35206725, 35207126, 35207127, 35207128, 35207129, 35207130, 35207131, 35207136, 35207137, 35207138, 35207139, 35207142, 35207143, 35207144, 35207145, 35207146, 35207147, 35207148, 35207149, 35207150, 35207151, 35207152, 35207153,
 35207154, 35207155, 35207156, 35207157, 35207158, 35207159, 35207160, 35207161, 35207162, 35207163, 35207164, 35207165, 35207169, 35207170, 35207171, 35207172, 35207173, 35207175, 35207176, 35207177, 35207190, 35207191, 35207192, 35207220, 35207221, 35207222, 35207223, 35207224, 35207726, 35207727, 35208023, 35208024, 35208025, 35208968, 35208969, 35208970, 35208971, 35209005, 35209006, 35209007, 35209008, 35209009, 35209010, 35209011, 35209013, 35211360, 35211361, 35211362, 35211363, 35211364, 35211365, 35211367, 35211368, 37200319, 37200320, 37200321, 37200322, 37200680, 37402461, 37402462, 37402466, 37402467, 37402468, 37402470, 37402471, 37402472, 45533065, 45533073, 45533074, 45533076, 45533077, 45533078, 45533080, 45533085, 45533088, 45533089, 45533626, 45533627, 45533628, 45533630, 45533867, 45533868, 45533939, 45533940, 45533941, 45533942, 45534323, 45534325, 45534326, 45534456, 45536716, 45536718, 45536719, 45536721, 45536727, 45536733, 45538005, 45538006, 45538007, 45538009,
 45538019, 45538021, 45538022, 45538023, 45538024, 45538027, 45538029, 45538030, 45538031, 45538032, 45538033, 45538034, 45538035, 45538036, 45538037, 45538038, 45538039, 45538040, 45538041, 45538042, 45538043, 45538046, 45538053, 45538061, 45538062, 45538063, 45538064, 45538069, 45538143, 45538563, 45538564, 45538565, 45538566, 45538567, 45538788, 45538789, 45538866, 45538867, 45538868, 45539222, 45539223, 45539225, 45539346, 45539347, 45541520, 45542708, 45542710, 45542791, 45542802, 45542803, 45542804, 45542805, 45542807, 45542808, 45542809, 45542810, 45542812, 45542813, 45542814, 45542818, 45542819, 45542820, 45542821, 45542829, 45542830, 45542831, 45542834, 45542838, 45542839, 45542841, 45542842, 45542843, 45542844, 45542857, 45542911, 45542912, 45543355, 45543356, 45543357, 45543359, 45543360, 45543612, 45543613, 45543689, 45543690, 45544038, 45544039, 45544040, 45544041, 45544149, 45546347, 45546348, 45546349, 45546351, 45546353, 45546354, 45546355, 45546356, 45546358, 45546362,
 45547442, 45547675, 45547679, 45547687, 45547689, 45547690, 45547691, 45547692, 45547693, 45547696, 45547703, 45547704, 45547706, 45547707, 45547708, 45547716, 45547763, 45548182, 45548183, 45548185, 45548186, 45548187, 45548188, 45548443, 45548444, 45548503, 45548850, 45548851, 45548852, 45548853, 45548854, 45548855, 45551158, 45551159, 45551162, 45551165, 45552212, 45552357, 45552431, 45552432, 45552436, 45552438, 45552439, 45552442, 45552443, 45552444, 45552445, 45552446, 45552447, 45552448, 45552449, 45552450, 45552451, 45552452, 45552453, 45552454, 45552460, 45552463, 45552477, 45552479, 45552489, 45552498, 45552499, 45552551, 45552964, 45552965, 45552966, 45552969, 45552970, 45552971, 45553204, 45553206, 45553271, 45553617, 45553618, 45553620, 45555951, 45555954, 45555955, 45557154, 45557155, 45557169, 45557170, 45557171, 45557172, 45557173, 45557174, 45557175, 45557176, 45557177, 45557178, 45557191, 45557192, 45557200, 45557201, 45557686, 45557687, 45557688, 45557690, 45557692,
 45557693, 45557694, 45557901, 45557902, 45557903, 45557992, 45557993, 45557994, 45558348, 45558349, 45558350, 45560702, 45560707, 45560708, 45560709, 45561923, 45561924, 45561925, 45561926, 45562002, 45562003, 45562005, 45562010, 45562011, 45562013, 45562014, 45562015, 45562021, 45562024, 45562025, 45562030, 45562031, 45562032, 45562033, 45562038, 45562109, 45562522, 45562523, 45562524, 45562525, 45562526, 45562527, 45562528, 45562530, 45562837, 45562838, 45562839, 45562840, 45563196, 45563197, 45563200, 45565528, 45566540, 45566702, 45566775, 45566776, 45566777, 45566778, 45566779, 45566780, 45566781, 45566792, 45566793, 45566796, 45566797, 45566798, 45566799, 45566800, 45566802, 45566804, 45566805, 45566808, 45566809, 45566810, 45566818, 45566819, 45566820, 45566825, 45566826, 45566827, 45566828, 45566829, 45566839, 45566840, 45566841, 45566906, 45566907, 45567337, 45567338, 45567339, 45567340, 45567341, 45567342, 45567343, 45567344, 45567578, 45567579, 45567580, 45567659, 45567660,
 45568004, 45568006, 45568007, 45568009, 45568135, 45568136, 45570391, 45570392, 45570393, 45570397, 45571635, 45571707, 45571708, 45571709, 45571724, 45571725, 45571726, 45571727, 45571730, 45571731, 45571732, 45571733, 45571734, 45571735, 45571736, 45571738, 45571741, 45571742, 45571743, 45571753, 45571756, 45571761, 45571768, 45572245, 45572246, 45572247, 45572248, 45572249, 45572250, 45572251, 45572252, 45572253, 45572254, 45572255, 45572256, 45572257, 45572258, 45572471, 45572552, 45572903, 45572904, 45572905, 45572906, 45572908, 45572909, 45572912, 45575247, 45575249, 45576406, 45576493, 45576494, 45576495, 45576497, 45576505, 45576507, 45576510, 45576511, 45576512, 45576514, 45576516, 45576519, 45576521, 45576526, 45576527, 45576528, 45576530, 45576538, 45576539, 45576540, 45576541, 45576545, 45576546, 45576547, 45577011, 45577013, 45577014, 45577015, 45577016, 45577017, 45577018, 45577019, 45577020, 45577021, 45577022, 45577025, 45577358, 45577359, 45577672, 45577673, 45577674,
 45577675, 45577676, 45577677, 45577678, 45577796, 45580099, 45580101, 45580104, 45580105, 45580108, 45581151, 45581324, 45581404, 45581405, 45581418, 45581421, 45581424, 45581425, 45581429, 45581430, 45581431, 45581432, 45581434, 45581435, 45581438, 45581439, 45581441, 45581442, 45581455, 45581461, 45581519, 45581919, 45581920, 45581921, 45581922, 45581924, 45581925, 45581926, 45581928, 45581930, 45582174, 45582175, 45582176, 45582247, 45582248, 45582249, 45582250, 45582251, 45582591, 45582592, 45582593, 45582595, 45582596, 45582597, 45582598, 45582601, 45582713, 45584902, 45584904, 45584906, 45586115, 45586187, 45586188, 45586189, 45586190, 45586192, 45586205, 45586207, 45586208, 45586212, 45586213, 45586214, 45586216, 45586217, 45586219, 45586220, 45586227, 45586235, 45586243, 45586247, 45586752, 45586753, 45586755, 45586758, 45586759, 45586761, 45586762, 45586993, 45587064, 45587065, 45587066, 45587067, 45587068, 45587069, 45587399, 45587400, 45587401, 45589751, 45589755, 45589756,
 45589761, 45590768, 45591082, 45591083, 45591084, 45591087, 45591104, 45591107, 45591108, 45591110, 45591111, 45591113, 45591114, 45591115, 45591116, 45591122, 45591123, 45591124, 45591125, 45591126, 45591134, 45591135, 45591136, 45591201, 45591617, 45591618, 45591619, 45591622, 45591623, 45591624, 45591934, 45591935, 45591936, 45592311, 45592313, 45592314, 45592315, 45592317, 45592419, 45594616, 45594617, 45594622, 45594627, 45595765, 45595766, 45595845, 45595846, 45595847, 45595851, 45595863, 45595864, 45595865, 45595866, 45595867, 45595868, 45595871, 45595872, 45595874, 45595875, 45595876, 45595877, 45595878, 45595879, 45595880, 45595884, 45595885, 45595891, 45595892, 45595893, 45595901, 45595903, 45596355, 45596356, 45596357, 45596358, 45596359, 45596360, 45596361, 45596362, 45596592, 45596593, 45596594, 45596595, 45597053, 45597054, 45597055, 45597056, 45597057, 45599435, 45599437, 45599442, 45599443, 45599447, 45599450, 45600375, 45600606, 45600692, 45600693, 45600694, 45600696,
 45600707, 45600708, 45600709, 45600711, 45600712, 45600714, 45600715, 45600719, 45600720, 45600725, 45600726, 45600727, 45600728, 45600729, 45600746, 45600747, 45601199, 45601200, 45601201, 45601202, 45601203, 45601204, 45601205, 45601206, 45601207, 45601208, 45601479, 45601480, 45601545, 45601546, 45601547, 45601910, 45604250, 45604253, 45604255, 45604256, 45604260, 45605452, 45605456, 45605466, 45605467, 45605468, 45605469, 45605471, 45605472, 45605473, 45605474, 45605475, 45605476, 45605482, 45605483, 45605507, 45605508, 45605968, 45605969, 45605970, 45605971, 45605973, 45605974, 45605975, 45606323, 45606324, 45606669, 45606670, 45606671, 45606672, 45606673, 45606674, 45606676, 45606677, 45606678, 45606679, 45606680, 45606681, 45609013, 45609015, 45609018, 45609019, 725216, 725217, 725219, 725221, 725222, 725224, 725226, 725228, 725229, 725231, 725234, 725235, 725236, 725237, 725271, 725272, 725273, 725274, 725275, 725277, 725278, 725279, 725280, 725284, 725285, 725287, 725288,
 725289, 725290, 725292, 725293, 725295, 725450, 725538, 725539, 725542, 725543, 725550, 725552, 725563, 725588, 725592, 766349, 766412, 766413, 766414, 766432, 766450, 766462, 766464, 766468, 766475, 766479, 766491, 766493)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 0 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (21604254)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria ) ))) c_occurrence 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` visit 
                ON v.visit_concept_id = visit.concept_id"""

export_bigquery_result_to_bucket(
    sql_query=dataset_18235021_condition_sql, 
    file_name="conditions" 
)

In [ ]:
!gsutil ls -lh {os.getenv("WORKSPACE_BUCKET")}/amia/*.csv

In [ ]:
import pandas
import os

# This query represents dataset "survey_amia" for domain "survey" and was generated for All of Us Controlled Tier Dataset v8
dataset_27477800_survey_sql = """
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.survey,
        answer.question_concept_id,
        answer.question,
        answer.answer_concept_id,
        answer.answer  
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.ds_survey` answer   
    WHERE
        (
            question_concept_id IN (1384490, 1384518, 1384625, 1585370, 1585375, 1585386, 1585389, 1585402, 1585711, 1585717, 1585723, 1585729, 1585735, 1585741, 1585747, 1585754, 1585760, 1585838, 1585845, 1585852, 1585879, 1585886, 1585889, 1585892, 1585899, 1585940, 1585952, 1586150, 1586151, 1586152, 1586153, 1586154, 1586155, 1586156, 43528663, 43528665, 43528841, 43528842, 43528843, 43528847, 43529899, 43529903, 43529904, 43529905, 43529906, 43529976, 43529977, 43530268, 43530356, 43530357, 43530358, 43530362, 43530409, 43530410, 43530411, 43530412, 43530413, 43530415, 43530416, 43530417, 43530418, 43530437, 43530438, 43530439, 43530514, 43530515, 43530516, 43530520, 43530557, 43530559, 43530562, 43530583, 43530584, 43530585, 43530588, 43530593, 43530594, 43530595, 836803, 836804, 836805, 836827, 836850, 903575, 903578) 
            OR question_concept_id IN (SELECT
                DISTINCT concept_id                         
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c                         
            JOIN
                (SELECT
                    CAST(cr.id as string) AS id                               
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr                               
                WHERE
                    concept_id IN (1585855, 40192389)                               
                    AND domain_id = 'SURVEY') a 
                    ON (c.path like CONCAT('%', a.id, '.%'))                         
            WHERE
                domain_id = 'SURVEY'                         
                AND type = 'PPI'                         
                AND subtype = 'QUESTION')
        )  
        AND (
            answer.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    criteria.person_id 
                FROM
                    (SELECT
                        DISTINCT person_id, entry_date, concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                    WHERE
                        (concept_id IN(SELECT
                            DISTINCT ca.descendant_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                        JOIN
                            (SELECT
                                DISTINCT c.concept_id       
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id             
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                WHERE
                                    concept_id IN (21604254)             
                                    AND full_text LIKE '%_rank1]%'       ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) b 
                                ON (ca.ancestor_id = b.concept_id)) 
                            AND is_standard = 1)) criteria ) ))"""
export_bigquery_result_to_bucket(
    sql_query=dataset_27477800_survey_sql, 
    file_name="surveys"
)

In [ ]:
!gsutil ls -lh {os.getenv("WORKSPACE_BUCKET")}/amia/*.csv